## Step 10 — create pre-BEAM diagnostic tables
**# of cells in notebook:** 3

**Purpose:** Create diagnostic information that helps identify blocks that may benefit from an alternative BEAM-based splitting approach. Buildings within each selected source block are buffered by 2 m, 3 m, and 4 m and dissolved into contiguous building groups. The number of resulting groups is summarized and combined with source-block characteristics for manual review. The notebook does not automatically determine which blocks proceed to the BEAM workflow; the user makes that decision based on the diagnostic table and visual inspection.

**Input:**

- the block-specific workspaces created in Step 2, each containing:
  - `buildings`
- `heterogeneous_largePop_blocks` from Step 1, including:
  - `LargePop`
  - `building_area_sum`
  - `block_area_sqm`
  - `population`

**Output:**

Within each block-specific geodatabase:

- `buildings_bfr_2m`
- `buildings_bfr_2m_dslv`
- `buildings_bfr_3m`
- `buildings_bfr_3m_dslv`
- `buildings_bfr_4m`
- `buildings_bfr_4m_dslv`

At the base block directory:

- `buildings_buffer_dissolve_counts.csv`
- `buildings_buffer_dissolve_counts_joined.csv`

**Main logic:**

**Cell 1 — Create buffered and dissolved building groups**

1. Iterates through the block-specific workspaces created in Step 2.
2. Reads the `buildings` feature class from each block-specific geodatabase.
3. Buffers each building by 2 m, 3 m, and 4 m using planar buffers.
4. For each buffer distance, dissolves touching or overlapping buffers into singlepart features.
5. Writes the buffer and dissolved-buffer layers back to the block-specific geodatabase.
6. Reports the number of processed, skipped, and failed block geodatabases.

**Cell 2 — Count dissolved building groups**

1. Iterates through the block-specific workspaces.
2. Counts features in `buildings_bfr_2m_dslv`, `buildings_bfr_3m_dslv`, and `buildings_bfr_4m_dslv`.
3. Writes one row per source block to `buildings_buffer_dissolve_counts.csv`.

**Cell 3 — Join source-block characteristics**

1. Reads the count table from Cell 2.
2. Extracts a consistent numeric block identifier.
3. Reads `LargePop`, `building_area_sum`, `block_area_sqm`, and `population` from `heterogeneous_largePop_blocks`.
4. Joins those source-block attributes to the buffer/dissolve counts.
5. Reports unmatched records and writes `buildings_buffer_dissolve_counts_joined.csv`.

**Use of the output:** The joined table is a diagnostic/manual decision aid. The user reviews the table together with visual inspection of candidate blocks and manually specifies the blocks to process in Notebook 11.


In [ ]:
import arcpy
import csv
import os
import re
import traceback
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# USER SETTINGS
# ------------------------------------------------------------

# This repository snapshot is configured to Johannesburg for consistency with
# the surrounding Steps 6–15. Change these paths to run the same workflow for
# another city.

base_dir = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks"
)

source_blocks_gdb = Path(
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
)
source_blocks_layer = "heterogeneous_largePop_blocks"
source_blocks_fc = os.path.join(str(source_blocks_gdb), source_blocks_layer)

buffer_distances = [2, 3, 4]

counts_csv = base_dir / "buildings_buffer_dissolve_counts.csv"
joined_csv = base_dir / "buildings_buffer_dissolve_counts_joined.csv"

join_fields = [
    "LargePop",
    "building_area_sum",
    "block_area_sqm",
    "population",
]


# ------------------------------------------------------------
# ENVIRONMENT
# ------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

if not base_dir.is_dir():
    raise FileNotFoundError(f"Base directory does not exist:\n{base_dir}")

if not arcpy.Exists(source_blocks_fc):
    raise FileNotFoundError(
        "Source heterogeneous/LargePop blocks layer does not exist:\n"
        f"{source_blocks_fc}"
    )

print("Inputs verified.")
print(f"Base block directory: {base_dir}")
print(f"Source blocks:        {source_blocks_fc}")


# ------------------------------------------------------------
# CELL 1 — CREATE BUFFERED / DISSOLVED BUILDING GROUPS
# ------------------------------------------------------------

processed_gdbs = 0
skipped_gdbs = 0
failed_gdbs = 0

for block_folder in sorted(base_dir.iterdir()):

    if not block_folder.is_dir():
        continue

    # Process only block folders like _397, _9872, etc.
    if not re.fullmatch(r"_\d+", block_folder.name):
        continue

    expected_gdb_name = f"{block_folder.name}.gdb"
    gdb_path = block_folder / expected_gdb_name

    if not arcpy.Exists(str(gdb_path)):
        print("\n------------------------------------------------------------")
        print(f"SKIPPING: no expected geodatabase found for {block_folder.name}")
        print(f"Expected: {gdb_path}")
        skipped_gdbs += 1
        continue

    buildings_fc = os.path.join(str(gdb_path), "buildings")

    if not arcpy.Exists(buildings_fc):
        print("\n------------------------------------------------------------")
        print(f"SKIPPING: buildings feature class not found in {gdb_path}")
        skipped_gdbs += 1
        continue

    print("\n============================================================")
    print(f"Processing folder: {block_folder.name}")
    print(f"GDB: {gdb_path}")
    print(f"Buildings: {buildings_fc}")

    try:
        building_count = int(arcpy.management.GetCount(buildings_fc)[0])
        print(f"Input building features: {building_count}")

        if building_count == 0:
            print("No buildings found. Skipping buffer/dissolve for this block.")
            skipped_gdbs += 1
            continue

        for dist in buffer_distances:

            print("\n------------------------------------------------------------")
            print(f"Buffer distance: {dist} meters")

            out_buffer = os.path.join(str(gdb_path), f"buildings_bfr_{dist}m")
            out_dissolve = os.path.join(str(gdb_path), f"buildings_bfr_{dist}m_dslv")

            # Buffer each building separately.
            if arcpy.Exists(out_buffer):
                arcpy.management.Delete(out_buffer)

            arcpy.analysis.Buffer(
                in_features=buildings_fc,
                out_feature_class=out_buffer,
                buffer_distance_or_field=f"{dist} Meters",
                line_side="FULL",
                line_end_type="ROUND",
                dissolve_option="NONE",
                dissolve_field=None,
                method="PLANAR",
            )

            buffer_count = int(arcpy.management.GetCount(out_buffer)[0])
            print(f"Created: {out_buffer}")
            print(f"Buffered features: {buffer_count}")

            # Dissolve touching/overlapping buffers into singlepart groups.
            if arcpy.Exists(out_dissolve):
                arcpy.management.Delete(out_dissolve)

            arcpy.management.Dissolve(
                in_features=out_buffer,
                out_feature_class=out_dissolve,
                dissolve_field=None,
                statistics_fields=None,
                multi_part="SINGLE_PART",
                unsplit_lines="DISSOLVE_LINES",
            )

            dissolve_count = int(arcpy.management.GetCount(out_dissolve)[0])
            print(f"Created: {out_dissolve}")
            print(f"Dissolved singlepart features: {dissolve_count}")

        processed_gdbs += 1

    except Exception as e:
        print("\nFAILED")
        print(f"Folder: {block_folder.name}")
        print(f"GDB: {gdb_path}")
        print(str(e))
        print(traceback.format_exc())
        failed_gdbs += 1


print("\n============================================================")
print("BUFFER / DISSOLVE STEP COMPLETE")
print(f"Processed GDBs: {processed_gdbs}")
print(f"Skipped GDBs:   {skipped_gdbs}")
print(f"Failed GDBs:    {failed_gdbs}")
print("============================================================")


In [ ]:
# ------------------------------------------------------------
# CELL 2 — COUNT DISSOLVED BUILDING GROUPS
# ------------------------------------------------------------

layers_to_count = [
    f"buildings_bfr_{dist}m_dslv"
    for dist in buffer_distances
]

rows = []

for block_folder in sorted(base_dir.iterdir()):

    if not block_folder.is_dir():
        continue

    if not re.fullmatch(r"_\d+", block_folder.name):
        continue

    block_id = block_folder.name.lstrip("_")
    gdb_path = block_folder / f"{block_folder.name}.gdb"

    print(f"\nProcessing block: {block_id}")

    row = {"block_id": block_id}

    for layer_name in layers_to_count:
        row[f"{layer_name}_count"] = None

    if not arcpy.Exists(str(gdb_path)):
        print(f"  Missing geodatabase: {gdb_path}")
        rows.append(row)
        continue

    for layer_name in layers_to_count:

        fc_path = os.path.join(str(gdb_path), layer_name)
        output_field = f"{layer_name}_count"

        if arcpy.Exists(fc_path):
            try:
                count = int(arcpy.management.GetCount(fc_path)[0])
                row[output_field] = count
                print(f"  {layer_name}: {count}")
            except Exception as e:
                print(f"  ERROR counting {layer_name}: {e}")
                row[output_field] = "ERROR"
        else:
            print(f"  Missing layer: {layer_name}")
            row[output_field] = "MISSING"

    rows.append(row)


fieldnames = [
    "block_id",
    *[f"{layer_name}_count" for layer_name in layers_to_count],
]

with open(counts_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("\nDone.")
print(f"Output CSV written to:\n{counts_csv}")


In [ ]:
# ------------------------------------------------------------
# CELL 3 — JOIN SOURCE-BLOCK CHARACTERISTICS
# ------------------------------------------------------------

def extract_numeric_block_id(value):
    """
    Convert values such as blk_14484 or 14484 to a string numeric ID.
    """
    if value is None:
        return None

    text = str(value).strip()
    match = re.search(r"(\d+)$", text)

    if match:
        return match.group(1)

    return None


# Read the count table from Cell 2.
df = pd.read_csv(counts_csv)
df.columns = [c.strip() for c in df.columns]

if "block_id" not in df.columns:
    raise ValueError(
        f"'block_id' not found in CSV. Available columns: {list(df.columns)}"
    )

df["join_block_id"] = df["block_id"].apply(extract_numeric_block_id)

print(f"CSV rows: {len(df):,}")
print("CSV columns:")
print(df.columns.tolist())


# Read selected attributes from the Step 1 source-block layer.
required_gdb_fields = ["block_id"] + join_fields

available_fields = [f.name for f in arcpy.ListFields(source_blocks_fc)]
missing = [f for f in required_gdb_fields if f not in available_fields]

if missing:
    raise ValueError(
        "Missing required fields in source blocks layer:\n"
        + "\n".join(missing)
    )

rows = []

with arcpy.da.SearchCursor(source_blocks_fc, required_gdb_fields) as cursor:
    for cursor_row in cursor:
        row_dict = dict(zip(required_gdb_fields, cursor_row))
        row_dict["join_block_id"] = extract_numeric_block_id(
            row_dict["block_id"]
        )
        rows.append(row_dict)

blocks_df = pd.DataFrame(rows)
blocks_df = blocks_df.rename(columns={"block_id": "gdb_block_id"})

print(f"GDB block rows: {len(blocks_df):,}")


# Join source-block attributes to the diagnostic counts.
joined = df.merge(
    blocks_df[["join_block_id", "gdb_block_id"] + join_fields],
    on="join_block_id",
    how="left",
)


# Diagnostics.
n_missing = int(joined["gdb_block_id"].isna().sum())

print(f"Joined rows: {len(joined):,}")
print(f"Rows with no GDB match: {n_missing:,}")

if n_missing > 0:
    print("\nExample unmatched CSV block_ids:")
    print(
        joined.loc[
            joined["gdb_block_id"].isna(),
            "block_id",
        ]
        .head(20)
        .tolist()
    )

joined = joined.drop(columns=["join_block_id"])
joined.to_csv(joined_csv, index=False)

print("\nDone.")
print(f"Output written to:\n{joined_csv}")
print(
    "\nThis table is a diagnostic/manual decision aid. "
    "Review it together with visual inspection before entering "
    "the selected block IDs in Notebook 11."
)
